# 🌐 MDAAR Free BYOK Multi-Provider Router (2nd Colab / Self-Hosted Hub)
### راوتر مفاتيح مجاني بلا حدود للجمع بين Gemini وGroq وOpenRouter وMistral وCerebras في رابط واحد
---
**المزايا:**
- واجهة تفاعلية نظيفة تُظهر فقط حقول المزودين الذين تفعّلهم وتُخفي البقية، وتفعّل تلقائياً أي مزود محفوظ في **Colab Secrets (🔑)**.
- تدوير المفاتيح والانتقال التلقائي عند أي `429`/`404` عبر رابط آمن محمي بـ Bearer Token.

### 🔑 أسرار كولاب المدعومة
| اسم المتغير / Secret | الوظيفة والافتراضي |
|:---|:---|
| `MDAAR_ROUTER_TOKEN` | توكن الحماية Bearer Token لربط النفق بدفتر MDAAR بأمان |
| `MDAAR_ROUTER_PORT` / `MDAAR_ROUTER_COOLDOWN` | المنفذ (`8787`) ومهلة التهدئة (`45` ثانية) |
| `GEMINI_API_KEY` أو `MDAAR_ROUTER_GEMINI_KEYS` | مفاتيح Gemini (`MDAAR_ROUTER_GEMINI_MODELS` افتراضياً `auto-free-chain`) |
| `GROQ_API_KEY` أو `MDAAR_ROUTER_GROQ_KEYS` | مفاتيح Groq (`MDAAR_ROUTER_GROQ_MODELS`) |
| `OPENROUTER_API_KEY` أو `MDAAR_ROUTER_OPENROUTER_KEYS` | مفاتيح OpenRouter (`MDAAR_ROUTER_OPENROUTER_MODELS`) |
| `MISTRAL_API_KEY` أو `MDAAR_ROUTER_MISTRAL_KEYS` | مفاتيح Mistral (`MDAAR_ROUTER_MISTRAL_MODELS`) |
| `CEREBRAS_API_KEY` أو `MDAAR_ROUTER_CEREBRAS_KEYS` | مفاتيح Cerebras (`MDAAR_ROUTER_CEREBRAS_MODELS`) |
| `MDAAR_ROUTER_CUSTOM_BASE_URL` / `_KEYS` / `_MODELS` | مسار ومفاتيح ومودلات أي مزود OpenAI مخصص |

In [ ]:
#@title 1. تثبيت MDAAR (وضع خفيف بدون GPU) و cloudflared { display-mode: "form" }
import subprocess

print('⏳ جاري تثبيت حزمة MDAAR (تشغيل قياسي بدون مكتبات GPU)...')
subprocess.run(['pip', 'install', '-q', '--upgrade', 'git+https://github.com/braadiv/MDAAR.git'], check=True)

cf_check = subprocess.run(['which', 'cloudflared'], capture_output=True, text=True)
if not cf_check.stdout.strip():
    print('⏳ تثبيت cloudflared لنفق الربط...')
    subprocess.run(['wget', '-q', 'https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb'], check=True)
    subprocess.run(['dpkg', '-i', 'cloudflared-linux-amd64.deb'], check=True)

from mdaar_pipeline import __version__, FREE_GEMINI_FALLBACK_CHAIN
print(f'🎉 تم تجهيز MDAAR FreeRouterServer v{__version__}!')
print('💡 سلسلة Gemini المجانية الجاهزة:', ', '.join(FREE_GEMINI_FALLBACK_CHAIN))

In [ ]:
#@title 2. لوحة اختيار المزودين وإدخال المفاتيح (تفاعلية) { display-mode: "form" }
from mdaar_pipeline import launch_free_router_ui
router_ui = launch_free_router_ui(lang='ar', namespace=globals())

In [ ]:
#@title 3. تشغيل الراوتر وفتح نفق Cloudflare وطباعة بيانات الربط { display-mode: "form" }
import json, urllib.request

router_server = router_ui.apply()
public_root_url = router_server.serve_with_tunnel(timeout=30, stop_existing=True)

with urllib.request.urlopen(f'http://127.0.0.1:{router_server.bound_port}/health', timeout=5) as resp:
    health_ok = json.loads(resp.read().decode('utf-8'))

print('✅ فحص الخادم المحلي ناجح:', health_ok['status'], '| المسارات:', len(health_ok['routes']))
print('=' * 70)
if public_root_url:
    public_v1 = f'{public_root_url}/v1'
    print('🌐 بيانات الربط لدفتر MDAAR الرئيسي (OpenAI-Gateway):')
    print(f'   • Shared Gateway URL : {public_v1}')
    print(f'   • Model              : auto')
    print('   • Shared Gateway Key : توكن MDAAR_ROUTER_TOKEN (أو التوكن المولّد في اللوحة أعلاه)')
else:
    print('⚠️ تعذر استخراج رابط cloudflared، الخادم يعمل محلياً على:', router_server.base_url)
print('=' * 70)

In [ ]:
#@title 4. مراقبة الطلبات وحالة التبديل الحي (Keep-Alive & Stats) { display-mode: "form" }
MONITOR_MINUTES = 60 #@param {type:"integer"}
REFRESH_SECONDS = 30 #@param {type:"integer"}

import time, json

print(f'📡 جاري مراقبة الراوتر لمدة {MONITOR_MINUTES} دقيقة...')
end_time = time.time() + max(1, int(MONITOR_MINUTES)) * 60
try:
    while time.time() < end_time:
        stats = router_server.get_health_payload()['stats']
        print(
            f'[{time.strftime("%H:%M:%S")}] إجمالي={stats["total_requests"]} | '
            f'نجاح={stats["success_requests"]} | فشل={stats["failed_requests"]} | '
            f'تهدئة نشطة={stats["active_cooldowns"]} | حسب المزود={json.dumps(stats["successes_by_route"], ensure_ascii=False)}'
        )
        time.sleep(max(5, int(REFRESH_SECONDS)))
except KeyboardInterrupt:
    print('⏹️ تم إيقاف طباعة المراقبة (الخادم والنفق ما زالا يعملان بالخلفية).')

In [ ]:
#@title 5. إيقاف الراوتر والنفق بشكل نظيف (اختياري) { display-mode: "form" }
if 'router_server' in globals() and router_server is not None:
    router_server.stop()
    print('🛑 تم إيقاف خادم الراوتر ونفق Cloudflare بنجاح.')